# FM-LLM Colab bootstrap

Run these cells top to bottom on a **GPU runtime** (Runtime → Change runtime type → T4/A100 → Save).
Sets up: Google Drive persistent storage, GPU check, SSH tunnel for VS Code Remote-SSH, and the repo clone + deps.
See `colab/README.md` in the repo for the full workflow explanation.

## 1. Mount Google Drive and create persistent storage layout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/fm_llm'
for sub in ['data', 'checkpoints', 'outputs', 'hf_cache']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

print('Persistent storage ready at:', DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

## 2. Confirm GPU is actually attached

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU attached — check Runtime > Change runtime type'

## 3. Open an SSH tunnel for VS Code Remote-SSH

Requires a free ngrok authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

This prompts you for the token interactively (never hardcode it in the notebook).

In [ ]:
!pip install -q colab_ssh --upgrade

from getpass import getpass
ngrok_token = getpass('Paste your ngrok authtoken (input hidden): ')

from colab_ssh import launch_ssh, init_git
launch_ssh(ngrok_token, password='')  # password='' -> key-based; colab_ssh prints setup instructions
print('\nCopy the SSH config block above into your local ~/.ssh/config, then in VS Code:')
print('  Remote-SSH: Connect to Host... -> select the printed host')

## 4. Clone/pull the repo and install GPU dependencies

Fill in `GIT_REMOTE_URL` once the repo has been pushed to GitHub (this project has
not been pushed anywhere yet — see `colab/README.md`).

In [ ]:
GIT_REMOTE_URL = ''  # e.g. 'https://github.com/<you>/fm_llm.git'
assert GIT_REMOTE_URL, 'Set GIT_REMOTE_URL to your GitHub repo URL first'

import os
if not os.path.exists('/content/fm_llm'):
    !git clone {GIT_REMOTE_URL} /content/fm_llm
else:
    !cd /content/fm_llm && git pull

%cd /content/fm_llm
!pip install -q uv
# On Colab's default index, `torch` resolves to a CUDA build automatically —
# no need for the CPU-only index configured for local dev.
!uv sync

## 5. Point the project at Drive-backed storage + Hugging Face token

Sets env vars the training code should read (once the data/config modules
exist) instead of writing into the Colab VM's ephemeral local disk.

In [ ]:
import os
os.environ['FM_LLM_DATA_ROOT'] = f'{DRIVE_ROOT}/data'
os.environ['FM_LLM_CHECKPOINT_ROOT'] = f'{DRIVE_ROOT}/checkpoints'
os.environ['FM_LLM_OUTPUT_ROOT'] = f'{DRIVE_ROOT}/outputs'
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'  # so Llama-3.2-1B weights persist across sessions

from getpass import getpass
os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token (input hidden): ')

print('Environment ready. Data/checkpoints/outputs/HF cache all point into Google Drive.')